# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 63 |
| **Integrantes** | Jorge Andrés Ocampo Suárez |
| **Caso de estudio** | Wanderbricks |
| **🎥 Enlace al video** | *(pegar aquí — 9 a 12 minutos, mínimo 4 minutos por integrante)* |

**Link Dashboard:** https://dbc-f55c2dd5-72d4.cloud.databricks.com/dashboardsv3/01f1bbaf84c51dc18714992148b689eb/published?o=7474652310838956

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

bookings = spark.table(f"{CATALOGO}.wanderbricks.silver_bookings")
properties = spark.table(f"{CATALOGO}.wanderbricks.silver_properties")
hosts = spark.table("samples.wanderbricks.hosts")
reviews = spark.table(f"{CATALOGO}.wanderbricks.silver_reviews")

In [0]:
reviews = (spark.table(f"{CATALOGO}.wanderbricks.silver_reviews")
    .select("booking_id", F.col("rating").alias("rating_reserva")))

# --- Recorrido del dato: conteos en cada capa ---
bronce_bookings = spark.table(f"{CATALOGO}.wanderbricks.bronze_bookings").count()
plata_bookings = bookings.count()
print(f"bookings — bronce: {bronce_bookings:,} | plata: {plata_bookings:,} | descartados: {bronce_bookings - plata_bookings:,}")

In [0]:
# --- Capa oro: desempeño por anfitrión, con nombres en vez de IDs ---
gold_anfitriones = (bookings
    .filter(F.col("status") == "completed")
    .join(properties, "property_id")
    .join(hosts.select("host_id", "name"), "host_id")  # solo lo que necesitamos de hosts
    .join(reviews, "booking_id", "left")
    .groupBy("host_id", "name")
    .agg(
        F.sum("total_amount").alias("ingresos_totales"),
        F.countDistinct("property_id").alias("numero_propiedades"),
        F.round(F.avg("rating_reserva"), 2).alias("rating_promedio")
    )
    .withColumnRenamed("name", "nombre_anfitrion")
    .orderBy(F.desc("ingresos_totales")))

filas_oro = gold_anfitriones.count()
print(f"oro — anfitriones resultantes: {filas_oro:,}")

gold_anfitriones.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.gold.desempeno_anfitriones")
display(spark.table(f"{CATALOGO}.gold.desempeno_anfitriones").orderBy(F.desc("ingresos_totales")).limit(10))

### Parte 1 — Cerrar el pipeline

**Recorrido del dato:**

| Capa | Tabla | Filas | Diferencia y por qué |
|---|---|---|---|
| Bronce | bookings | 72,247 | — |
| Plata | silver_bookings | 72,246 | Se descartó **1 registro** que violaba una de las dos reglas de calidad (`total_amount >= 0` y `check_out > check_in`). El número es bajo porque el dataset sintético de Wanderbricks ya viene mayormente consistente, no es una falla de las reglas, es una señal de que la fuente tiene buena calidad de origen. |
| Oro | desempeno_anfitriones | 2,318 | Se pasa de nivel-reserva a nivel-anfitrión: cada una de las filas de plata (agrupadas por `status = 'completed'`) colapsa en una sola fila por `host_id`. 2,318 anfitriones distintos generaron al menos una reserva completada. |

**Tabla oro `gold.desempeno_anfitriones`:** una fila por anfitrión (2,318 en total), con su nombre, ingresos totales, número de propiedades administradas y rating promedio de sus huéspedes.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

page_views = spark.table(f"{CATALOGO}.wanderbricks.bronze_page_views")
users = spark.table(f"{CATALOGO}.wanderbricks.bronze_users").select("user_id", "country")
bookings = spark.table(f"{CATALOGO}.wanderbricks.silver_bookings")

gasto_por_usuario = (bookings
    .filter(F.col("status") == "completed")
    .groupBy("user_id")
    .agg(F.sum("total_amount").alias("gasto_total")))

def consulta_base_ea4():
    resultado = (page_views
        .join(users, "user_id")
        .join(gasto_por_usuario, "user_id", "left")
        .groupBy("country")
        .agg(
            F.count("view_id").alias("vistas_totales"),
            F.sum("gasto_total").alias("gasto_total_pais")
        ))
    resultado.collect()

# Plan de ejecución inicial
df_base = (page_views
    .join(users, "user_id")
    .join(gasto_por_usuario, "user_id", "left")
    .groupBy("country")
    .agg(F.count("view_id").alias("vistas_totales"), F.sum("gasto_total").alias("gasto_total_pais")))

df_base.explain(mode="formatted")

In [0]:
import time, statistics

def medir(funcion, repeticiones=3, etiqueta=""):
    """Ejecuta la función varias veces y reporta la mediana."""
    tiempos = []
    for i in range(repeticiones):
        inicio = time.time()
        funcion()
        tiempos.append(time.time() - inicio)
    mediana = statistics.median(tiempos)
    print(f"{etiqueta:35s} mediana {mediana:6.2f}s   corridas {[round(t,2) for t in tiempos]}")
    return mediana

# Calentamiento (se descarta) + 3 mediciones
consulta_base_ea4()  # calentamiento, no cuenta
t_base = medir(consulta_base_ea4, etiqueta="Base (join sin optimizar)")

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"

# Materializamos gasto_por_usuario UNA sola vez como tabla Delta
gasto_por_usuario = (spark.table(f"{CATALOGO}.wanderbricks.silver_bookings")
    .filter(F.col("status") == "completed")
    .groupBy("user_id")
    .agg(F.sum("total_amount").alias("gasto_total")))

gasto_por_usuario.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.wanderbricks.gold_gasto_por_usuario")

In [0]:
page_views = spark.table(f"{CATALOGO}.wanderbricks.bronze_page_views")
users = spark.table(f"{CATALOGO}.wanderbricks.bronze_users").select("user_id", "country")
gasto_materializado = spark.table(f"{CATALOGO}.wanderbricks.gold_gasto_por_usuario")

def consulta_optimizada_ea4():
    resultado = (page_views
        .join(users, "user_id")
        .join(gasto_materializado, "user_id", "left")
        .groupBy("country")
        .agg(
            F.count("view_id").alias("vistas_totales"),
            F.sum("gasto_total").alias("gasto_total_pais")
        ))
    resultado.collect()

# Plan de ejecución optimizado
df_opt = (page_views
    .join(users, "user_id")
    .join(gasto_materializado, "user_id", "left")
    .groupBy("country")
    .agg(F.count("view_id").alias("vistas_totales"), F.sum("gasto_total").alias("gasto_total_pais")))

df_opt.explain(mode="formatted")

In [0]:
consulta_optimizada_ea4()  # calentamiento
t_opt = medir(consulta_optimizada_ea4, etiqueta="Optimizada (agregación pre-materializada)")
print(f"\nMejora: {(1 - t_opt/t_base)*100:.1f}%")

### Parte 2 — Optimización medida

**Operación elegida:** vistas de página por país cruzadas con el gasto total en reservas completadas de cada usuario — join entre `bronze_page_views` (500K filas), `bronze_users` y una agregación de `silver_bookings`.

**Plan inicial:** ambos joins ya eran `PhotonBroadcastHashJoin` de fábrica, el optimizador decidió difundir tanto `users` como el resultado agregado de gasto sin que se le pidiera. El costo real no estaba en la estrategia de join, sino en que **la agregación de gasto por usuario se recalculaba desde cero en cada ejecución**

**Medición base:** mediana de 1.39s sobre 3 corridas (1.38s, 1.63s, 1.39s).

**Técnica aplicada — materialización de la agregación intermedia:** se calculó `gasto_por_usuario` una sola vez y se guardó como tabla Delta (`gold_gasto_por_usuario`), en vez de recalcularla dentro de la misma consulta cada vez que se ejecuta.

**Plan optimizado:** comparando ambos planes, la diferencia es exacta y localizada — los pasos 8 a 17 del plan base (`PhotonScan silver_bookings` → `Project` → `PhotonGroupingAgg` parcial → shuffle → `PhotonGroupingAgg` final) desaparecen por completo. En su lugar, el plan optimizado simplemente lee la tabla ya agregada (`PhotonScan gold_gasto_por_usuario`, pasos 8-11) y la difunde directamente. Se eliminó una etapa completa de agregación distribuida con su shuffle asociado, reemplazándola por una lectura directa de datos ya calculados.

**Medición optimizada:** mediana de 1.30s sobre 3 corridas (1.16s, 1.80s, 1.30s) — **mejora del 6.4%**.

**Por qué la mejora fue modesta y no dramática:** aunque se eliminó una etapa completa del plan, `silver_bookings` no es una tabla enorme (72,246 filas) — la agregación que se evitó ya era relativamente barata comparada con el escaneo de las 500K filas de `page_views`, que sigue presente en ambas versiones. La optimización elimina trabajo real, pero no era el cuello de botella dominante de la consulta.

**Cuándo esta misma optimización sería contraproducente:** materializar una agregación intermedia como tabla solo vale la pena cuando esa agregación (a) es costosa de recalcular y (b) se reutiliza en varias consultas o se refresca con poca frecuencia frente a cuánto se consulta. Si `silver_bookings` cambiara constantemente, `gold_gasto_por_usuario` quedaría desactualizada casi de inmediato, y mantenerla fresca exigiría un pipeline de refresco adicional cuyo costo operativo podría superar el 6.4% de mejora que se ganó aquí. También dejaría de servir si el número de usuarios creciera mucho: la tabla materializada solo se benefició de un *broadcast join* porque es pequeña, si creciera lo suficiente como para dejar de caber en memoria para difusión, Spark volvería a un shuffle join de todas formas, y el costo de mantener la materialización ya no se justificaría.

### Parte 3 — Cerrar el caso

**Pregunta orientadora: una empresa recibe datos de su portal web, Facebook, Instagram, TikTok y WhatsApp Business. ¿Bajo qué paradigma, con qué herramientas y con qué arquitectura debería analizarlos?**

El paradigma correcto es un **lakehouse**, por la misma razón que se justificó en la EA1 para Wanderbricks: estas cinco fuentes combinan datos con estructura fija (metadatos de campaña, IDs de usuario, timestamps) con datos genuinamente variables entre plataforma y plataforma (un mensaje de WhatsApp Business no tiene la misma forma que un comentario de TikTok o una interacción de Instagram). Forzar todo a un esquema relacional rígido desde el día uno obligaría a renegociar el modelo cada vez que una API cambie su formato de respuesta — algo que ya vimos con `clickstream` en la EA1, donde el campo `metadata` traía estructura anidada que un relacional puro habría tenido que normalizar de entrada. Un NoSQL puramente documental, en cambio, resolvería la variedad pero perdería las garantías ACID y el versionado que se necesitan para auditar campañas y gasto publicitario — las mismas garantías que se demostraron con `MERGE`, `DESCRIBE HISTORY` y evolución de esquema en la EA1, y que en la EA2 se tradujeron en gobierno real vía Unity Catalog (permisos por capa, linaje automático).

La arquitectura seguiría el mismo patrón de capas trabajado en todo el curso: **bronce** ingiere cada fuente tal como llega de su API respectiva (portal web, Meta Graph API para Facebook/Instagram, TikTok API, WhatsApp Business API), sin normalizar entre ellas — cada una con su propio esquema, igual que se hizo con `clickstream` frente a `bookings`. **Plata** aplica limpieza y un esquema común mínimo (usuario, fecha, plataforma, tipo de interacción) que sí permite comparar entre fuentes, igual que las reglas de calidad aplicadas a `silver_bookings` y `silver_lineitem`. **Oro** agrega por lo que el negocio realmente pregunta (alcance por plataforma, conversión por canal), tal como se hizo aquí con `gold.desempeno_anfitriones` y `gold_gasto_por_usuario`. Sobre el procesamiento, la EA3 y esta misma evidencia mostraron que un cruce entre fuentes de volumen desigual (aquí, `page_views` con 500K filas contra `bookings` con 72K) se resuelve con Spark distribuido y no con procesamiento de un solo nodo — con cinco plataformas sociales generando eventos constantemente, ese volumen solo crecería. Y sobre el modelo de despliegue, la comparación de la EA2 aplica igual: un PaaS como Databricks evita que el equipo tenga que administrar la infraestructura de ingesta y cómputo para cinco fuentes distintas, dejando el esfuerzo de ingeniería en las transformaciones y el análisis, que es donde realmente está el valor de negocio.

---

### Conclusiones del proyecto completo

El hilo conductor de las cuatro evidencias fue que ninguna decisión de diseño se sostuvo por sí sola — cada una se validó con evidencia concreta generada en la siguiente etapa. La elección del lakehouse en la EA1 se apoyó en argumentos teóricos (integridad de `bookings`, estructura anidada de `clickstream`), pero se confirmó en la práctica en la EA2 (permisos y linaje funcionando sin fricción) y en esta EA4, donde la capa oro se construyó sin necesidad de rediseñar nada de lo ya hecho en agosto.

El aprendizaje más importante, sin embargo, vino de los resultados que **no** salieron como se esperaba, y que el proyecto obligó a reportar con honestidad en vez de maquillar: en la EA3, el broadcast join fue contraproducente (-13.7%) por una restricción de la plataforma (sin caché en serverless); en esta EA4, la optimización sí funcionó, pero de forma modesta (6.4%) porque el cuello de botella real no era el que se asumía al principio (el join, ya optimizado de fábrica por Photon) sino un recálculo evitable. En ambos casos, medir con tres corridas y comparar planes de ejecución fue lo que permitió distinguir una mejora real de una ilusión, y eso es la diferencia central que planteaba el enunciado de esta evidencia entre usar Spark y entenderlo.

Si se empezara de nuevo, se documentaría desde la EA1 la convención exacta de nombres (el desajuste `bigdata_grupo063` vs `bigdata_grupo63` detectado en la EA2 costó tiempo de corrección evitable), y se definiría desde el principio del proyecto una única función `medir()` compartida entre notebooks, en vez de tener que redefinirla en cada evidencia nueva.

**Dashboard**
![](/Workspace/Users/jorge.ocampo@est.iudigital.edu.co/bigdata-2026b-g063/ea3/desempeno_anfitriones_wanderbricks_DASHBOARD.png)